In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer 
import torch

model_name = "/data/hoang/resources/models/Qwen/Qwen3-8B" 
tokenizer = AutoTokenizer.from_pretrained(model_name) 
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    # device_map="auto"
)

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
import os
 
# Path to your Who&When JSON file — edit this.
DATASET_DIR = "./ww/hand-crafted"
 
# Model names used in the evaluation plan.
MODEL_LLAMA = "/data/hoang/resources/models/meta-llama/Meta-Llama-3-8B-Instruct"
MODEL_QWEN  = "/data/hoang/resources/models/Qwen/Qwen3-8B" 
 
# Which model to load in this session.
MODEL_NAME = MODEL_QWEN  
 
DEVICE = "cuda:0"   # "cpu" for a CPU-only sanity pass (slow)

In [4]:
from utils.common import _load_json_data, _get_sorted_json_files, _extract_metadata
from gradnorm.data import Trajectory
from pathlib import Path

def load_dataset(
    path: str | Path,
    subset: str | None = None,
) -> list[Trajectory]:
    """Load the Who&When dataset from a JSON file.

    Parameters
    ----------
    path   : path to the data directory
    subset : optional filter — "algo" | "handcrafted".
             Pass None to return everything.

    Returns
    -------
    list[Trajectory]

    Expected JSON schema per item
    ------------------------------
    {
        "history":       [{"role": str, "content": str}, ...],
        "mistake_agent": str,
        "mistake_step":  str | int,   # parsed to int; 0-indexed
        "question_ID":   str,
        "level":         int,          # optional
        "subset":        str,          # optional; "algo" | "handcrafted"
        "question":      str           # optional
    }

    Notes
    -----
    If the JSON file does not contain a "subset" key (e.g., separate files per
    subset), supply the subset label via the `subset` argument *as a filter*
    only, or pre-tag the items before calling this function.
    """
    path = Path(path)
    filenames = _get_sorted_json_files(path)
    raw = [_load_json_data(path / filename) for filename in filenames]

    trajectories: list[Trajectory] = []
    for item in raw:
        traj = Trajectory(
            question_id   = item["question_ID"],
            history       = item["history"],
            mistake_agent = item["mistake_agent"],
            mistake_step  = int(item["mistake_step"]),
            level         = item.get("level", -1),
            subset        = item.get("subset", "unknown"),
            question      = item.get("question", ""),
        )
        if subset is None or traj.subset == subset:
            trajectories.append(traj)

    return trajectories

In [5]:
import statistics
trajs = load_dataset(DATASET_DIR)
print(f"Total trajectories loaded : {len(trajs)}")
 
# Subset breakdown
from collections import Counter
subset_counts = Counter(t.subset for t in trajs)
print("Subset breakdown:", dict(subset_counts))
 
# Level breakdown
level_counts = Counter(t.level for t in trajs)
print("Level breakdown:", dict(sorted(level_counts.items())))

hc_lengths   = [len(t.history) for t in trajs]

print("Handcraft step counts — "
      f"min={min(hc_lengths)}  max={max(hc_lengths)}  "
      f"mean={statistics.mean(hc_lengths):.1f}")

Total trajectories loaded : 58
Subset breakdown: {'unknown': 58}
Level breakdown: {-1: 28, 1: 10, 2: 16, 3: 4}
Handcraft step counts — min=5  max=130  mean=51.6


In [6]:
traj = max(trajs, key=lambda t: len(t.history))
print(f"question_id   : {traj.question_id}")
print(f"subset        : {traj.subset}")
print(f"level         : {traj.level}")
print(f"total steps   : {len(traj.history)}")
print(f"mistake_step  : {traj.mistake_step}  (0-indexed)")
print(f"mistake_agent : {traj.mistake_agent}")

# Unique roles appearing in this trajectory.
roles = [step["role"] for step in traj.history]
unique_roles = list(dict.fromkeys(roles))   # ordered, deduplicated
print("Roles in order of first appearance:", unique_roles)

# Inspect the mistake step itself.
ms = traj.mistake_step
print(f"--- Step {ms} (the mistake step) ---")
print(f"Role    : {traj.history[ms]['role']}")
print(f"Content : {traj.history[ms]['content'][:500]!r}...")

question_id   : 624cbf11-6a41-4692-af9c-36b3e5ca3130
subset        : unknown
level         : 2
total steps   : 130
mistake_step  : 24  (0-indexed)
mistake_agent : WebSurfer
Roles in order of first appearance: ['human', 'Orchestrator (thought)', 'Orchestrator (-> WebSurfer)', 'WebSurfer', 'Orchestrator (-> Assistant)', 'Assistant', 'Orchestrator (termination condition)']
--- Step 24 (the mistake step) ---
Role    : WebSurfer
Content : 'I scrolled down one page in the browser.\n\nHere is a screenshot of [Flavor Graveyard | Ben & Jerry’s](https://www.benjerry.com/flavors/flavor-graveyard). The viewport shows 13% of the webpage, and is positioned 49% down from the top of the page.\nAutomatic OCR of the page screenshot has detected the following text:\n\nSugar Plum\n\nTennessee Mud\n\nThe Wich\n\nThis is Nuts\n\nTurtle Soup\n\nTuskegee Chunk\n\nUrban Jumble\n\nVermonty Python\n\nWavy Gravy\n\nWhite Russian\n<Image>'...


In [10]:
# double run this cell

from utils.graph import derive_llm_inputs, get_dependency_dict
from gradnorm.data import select_context

import gradnorm.data as data_module
 
def _select_from_dependency(history, step_idx):
    deps = get_dependency_dict(derive_llm_inputs(history))
    return deps[step_idx]
 
_original_select_context = data_module.select_context
 
data_module.select_context = _select_from_dependency
print("After monkey-patch, context for last step:", data_module.select_context(traj.history, step_idx=56))
 
late_step = len(traj.history) - 1
ctx_for_late = select_context(traj.history, late_step)
print(f"Context indices for step {late_step} (last step): {len(ctx_for_late)} turns")

After monkey-patch, context for last step: [0, 39, 41, 43, 45, 47, 49, 51, 53, 55]
Context indices for step 129 (last step): 7 turns


In [11]:
from gradnorm.data import _serialize_turns
 
ctx_indices = select_context(traj.history, ms)   # context for the mistake step
ctx_text    = _serialize_turns(traj.history, ctx_indices)
 
print(ctx_text[:500])
print("...")

[human] - Step 0: What's the last line of the rhyme under the flavor name on the headstone visible in the background of the photo of the oldest flavor's headstone in the Ben & Jerry's online flavor graveyard as of the end of 2022?

[Orchestrator (thought)] - Step 1: Initial plan:

We are working to address the following user request:

What's the last line of the rhyme under the flavor name on the headstone visible in the background of the photo of the oldest flavor's headstone in the Ben & Jerry
...


In [12]:
from transformers import AutoTokenizer
 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokeniser loaded: {MODEL_NAME}")
print(f"  Vocab size  : {tokenizer.vocab_size}")
print(f"  Chat template present: {tokenizer.chat_template is not None}")
 
# %%
from gradnorm.data import build_context
 
# Score the mistake step of our example trajectory.
encoded = build_context(traj.history, ms, tokenizer)

Tokeniser loaded: /data/hoang/resources/models/Qwen/Qwen3-8B
  Vocab size  : 151643
  Chat template present: True


In [13]:
input_ids = encoded["input_ids"]   # shape (1, seq_len)
ctx_len   = encoded["ctx_len"]
seq_len   = input_ids.shape[1]
step_len  = seq_len - ctx_len
 
print(f"Encoded step {ms}:")
print(f"  seq_len  = {seq_len}  tokens total")
print(f"  ctx_len  = {ctx_len}  tokens (context + assistant header)")
print(f"  step_len = {step_len} tokens (NTP loss computed over these)")

Encoded step 24:
  seq_len  = 3878  tokens total
  ctx_len  = 3748  tokens (context + assistant header)
  step_len = 130 tokens (NTP loss computed over these)


In [14]:
# Verify the boundary by decoding the last ctx_len tokens vs the first step tokens.
# The split should happen exactly at the assistant header boundary.
ctx_tokens  = tokenizer.convert_ids_to_tokens(input_ids[0, :ctx_len].tolist())
step_tokens = tokenizer.convert_ids_to_tokens(input_ids[0, ctx_len:ctx_len+10].tolist())
 
print("Last 8 context tokens  :", ctx_tokens[-8:])
print("First 10 step tokens   :", step_tokens)
 
# %%
# Demonstrate that ctx_len is computed via the prefix-only tokenisation trick.
# Tokenise [user_msg] with add_generation_prompt=True to get the exact prefix length.
ctx_indices  = select_context(traj.history, ms)
ctx_text_    = _serialize_turns(traj.history, ctx_indices)
user_msg     = {"role": "user", "content": ctx_text_}
 
prefix_ids = tokenizer.apply_chat_template(
    [user_msg],
    tokenize=True, add_generation_prompt=True, return_tensors="pt"
)

print(f"Prefix length from apply_chat_template: {prefix_ids["input_ids"].shape[1]}")
print(f"ctx_len stored in encoded dict        : {ctx_len}")
assert prefix_ids["input_ids"].shape[1] == ctx_len, "Mismatch — something is wrong."
print("ctx_len derivation VERIFIED.")
 
# %%
# Inspect a non-mistake step to see how ctx_len grows with trajectory depth.
for step_idx in [1, 5, ms, len(traj.history) - 1]:
    enc    = build_context(traj.history, step_idx, tokenizer)
    s_len  = enc["input_ids"].shape[1]
    c_len  = enc["ctx_len"]
    print(f"  step {step_idx:3d}  |  seq={s_len:6d}  ctx={c_len:6d}  step_toks={s_len - c_len:5d}")


Last 8 context tokens  : ['Ġif', 'Ġavailable', '.', '<|im_end|>', 'Ċ', '<|im_start|>', 'assistant', 'Ċ']
First 10 step tokens   : ['<think>', 'ĊĊ', '</think>', 'ĊĊ', 'I', 'Ġscrolled', 'Ġdown', 'Ġone', 'Ġpage', 'Ġin']
Prefix length from apply_chat_template: 3748
ctx_len stored in encoded dict        : 3748
ctx_len derivation VERIFIED.
  step   1  |  seq=   895  ctx=    66  step_toks=  829
  step   5  |  seq=  2468  ctx=  2180  step_toks=  288
  step  24  |  seq=  3878  ctx=  3748  step_toks=  130
  step 129  |  seq=  3839  ctx=  3729  step_toks=  110


## NTP Loss

In [15]:
import torch
from gradnorm.gradnorm import _ntp_loss
 
# Use a small, self-contained toy to verify mask alignment before touching the real model.
vocab, seq, ctx = 10, 6, 4
# Claim: step tokens are at positions ctx..seq-1 = [4, 5]
# After shift: logit[i] predicts token[i+1]
# So predicting step tokens uses logit positions [ctx-1, ..., seq-2] = [3, 4]
# => mask positions [0, 1, 2] in shift_labels (first ctx-1 = 3 positions).
 
torch.manual_seed(42)
logits    = torch.zeros(1, seq, vocab)
input_ids = torch.randint(0, vocab, (1, seq))
 
# Perfect predictions only for step positions (shifted: 3, 4 → predict tokens 4, 5)
for shifted_pos in range(ctx - 1, seq - 1):
    correct_next = input_ids[0, shifted_pos + 1].item()
    logits[0, shifted_pos, correct_next] = 20.0   # very high logit
 
loss_step_correct = _ntp_loss(logits, input_ids, ctx_len=ctx)
print(f"Loss when step tokens are perfect: {loss_step_correct.item():.6f}  (should be ≈ 0)")
 
# Perfect predictions only for context positions → high loss on step tokens.
logits2 = torch.zeros(1, seq, vocab)
for shifted_pos in range(ctx - 1):   # only context positions
    logits2[0, shifted_pos, input_ids[0, shifted_pos + 1].item()] = 20.0
 
loss_ctx_correct = _ntp_loss(logits2, input_ids, ctx_len=ctx)
print(f"Loss when only context is perfect: {loss_ctx_correct.item():.4f}  (should be ≈ log(vocab)={torch.log(torch.tensor(float(vocab))):.4f})")

Loss when step tokens are perfect: 0.000000  (should be ≈ 0)
Loss when only context is perfect: 2.3026  (should be ≈ log(vocab)=2.3026)


In [16]:
# Verify the position accounting once more with a diagram.
seq_, ctx_ = 6, 4
print("Token positions   :", list(range(seq_)))
print("Context positions :", list(range(ctx_)))
print("Step positions    :", list(range(ctx_, seq_)))
print()
print("Shifted positions (logit[i] → predicts token[i+1]):")
print("  All shifted    :", list(range(seq_ - 1)))
print("  Masked (ctx)   :", list(range(ctx_ - 1)))   # first ctx-1 = 3
print("  Active (step)  :", list(range(ctx_ - 1, seq_ - 1)))  # = [3, 4]
print("  These predict  :", [p + 1 for p in range(ctx_ - 1, seq_ - 1)])  # = [4, 5] ✓

Token positions   : [0, 1, 2, 3, 4, 5]
Context positions : [0, 1, 2, 3]
Step positions    : [4, 5]

Shifted positions (logit[i] → predicts token[i+1]):
  All shifted    : [0, 1, 2, 3, 4]
  Masked (ctx)   : [0, 1, 2]
  Active (step)  : [3, 4]
  These predict  : [4, 5]


In [17]:
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype = torch.bfloat16,
    device_map  = {"": 0},
)
model.eval()
 
n_params = sum(p.numel() for p in model.parameters())
print(f"Model    : {MODEL_NAME}")
print(f"Params   : {n_params/1e9:.2f}B")
print(f"Device   : {next(model.parameters()).device}")
print(f"Dtype    : {next(model.parameters()).dtype}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Model    : /data/hoang/resources/models/Qwen/Qwen3-8B
Params   : 8.19B
Device   : cuda:0
Dtype    : torch.bfloat16


In [18]:
def accounting(model):
    transformer = model.model
    print("Transformer body type  :", type(transformer).__name__)
    print("Number of layers       :", len(transformer.layers))
    print("Final layer type       :", type(transformer.layers[-1]).__name__)
    
    # %%
    # Attribute paths for each layer variant.
    final_layer = transformer.layers[-1]
    
    lm_head_W   = model.lm_head.weight
    out_proj_W  = final_layer.self_attn.o_proj.weight
    
    print(f"lm_head.weight    shape : {lm_head_W.shape}")     # (vocab, d_model)
    print(f"o_proj.weight     shape : {out_proj_W.shape}")    # (d_model, d_model)
    
    # All named modules in the final transformer layer.
    print("\nModules in final layer:")
    for name, mod in final_layer.named_modules():
        if hasattr(mod, "weight") and mod.weight is not None:
            print(f"  {name:<40} weight {tuple(mod.weight.shape)}")


accounting(model)

Transformer body type  : Qwen3Model
Number of layers       : 36
Final layer type       : Qwen3DecoderLayer
lm_head.weight    shape : torch.Size([151936, 4096])
o_proj.weight     shape : torch.Size([4096, 4096])

Modules in final layer:
  self_attn.q_proj                         weight (4096, 4096)
  self_attn.k_proj                         weight (1024, 4096)
  self_attn.v_proj                         weight (1024, 4096)
  self_attn.o_proj                         weight (4096, 4096)
  self_attn.q_norm                         weight (128,)
  self_attn.k_norm                         weight (128,)
  mlp.gate_proj                            weight (12288, 4096)
  mlp.up_proj                              weight (12288, 4096)
  mlp.down_proj                            weight (4096, 12288)
  input_layernorm                          weight (4096,)
  post_attention_layernorm                 weight (4096,)


In [19]:
from gradnorm.gradnorm import get_target_params
 
for variant in ["lm_head", "out_proj", "final_layer"]:
    params = get_target_params(model, variant)
    total  = sum(p.numel() for p in params)
    print(f"{variant:15s}  {len(params):2d} tensor(s)  {total/1e6:8.2f}M parameters")

lm_head           1 tensor(s)    622.33M parameters
out_proj          1 tensor(s)     16.78M parameters
final_layer      12 tensor(s)    815.28M parameters


In [20]:
encoded   = build_context(traj.history, ms, tokenizer)
input_ids = encoded["input_ids"].to(DEVICE)   # (1, seq_len)
ctx_len   = encoded["ctx_len"]

In [23]:
# ── Single forward pass — both losses from the same logits tensor ────────────
labels = input_ids.clone()
labels[:, :ctx_len] = -100

with torch.no_grad():
    out      = model(input_ids=input_ids, labels=labels, use_cache=False)
    hf_loss  = out.loss.item()
    our_loss = _ntp_loss(out.logits, input_ids, ctx_len).item()

print(f"HF built-in loss : {hf_loss:.6f}")
print(f"Our _ntp_loss    : {our_loss:.6f}")
print(f"Absolute diff    : {abs(hf_loss - our_loss):.2e}")
assert abs(hf_loss - our_loss) < 1e-5, "Losses differ — check mask alignment!"
print("MATCH ✓")

HF built-in loss : 1.865463
Our _ntp_loss    : 1.865463
Absolute diff    : 0.00e+00
MATCH ✓


In [24]:
from gradnorm.gradnorm import gradnorm_standard
 
# Use a short step to keep this fast.
# Pick the step right before the mistake step (usually short).
probe_step  = min(ms, 3)   # early step, short context
encoded_p   = build_context(traj.history, probe_step, tokenizer)
input_ids_p = encoded_p["input_ids"].to(DEVICE)
ctx_len_p   = encoded_p["ctx_len"]
 
print(f"Probing step {probe_step}  |  seq={input_ids_p.shape[1]}  ctx={ctx_len_p}")
print()
 
for variant in ["lm_head", "out_proj", "final_layer"]:
    score = gradnorm_standard(
        model=model, 
        input_ids=input_ids_p, 
        ctx_len=ctx_len_p,
        normalize=True, 
        layer_variant=variant
    )
    "{:.2e}".format(score)
    print(f"  standard / {variant:12s}  →  S = {'{:.2e}'.format(score)}")

Probing step 3  |  seq=300  ctx=267

  standard / lm_head       →  S = 1.34e-06
  standard / out_proj      →  S = 1.48e-04
  standard / final_layer   →  S = 2.47e-05


In [25]:
n_req_grad = sum(1 for p in model.parameters() if p.requires_grad)
n_stale    = sum(1 for p in model.parameters() if p.grad is not None)
print(f"params with requires_grad after scoring : {n_req_grad}  (should be 0)")
print(f"params with stale .grad after scoring   : {n_stale}  (should be 0)")

params with requires_grad after scoring : 0  (should be 0)
params with stale .grad after scoring   : 0  (should be 0)


In [26]:
def memory_accounting():
    torch.cuda.empty_cache()
    alloc_mb = torch.cuda.memory_allocated() / 1e6
    reserv_mb = torch.cuda.memory_reserved() / 1e6
    print(f"CUDA allocated : {alloc_mb:.1f} MB")
    print(f"CUDA reserved  : {reserv_mb:.1f} MB")
memory_accounting()

CUDA allocated : 17577.2 MB
CUDA reserved  : 17618.2 MB


In [27]:
from gradnorm.gradnorm import score_trajectory

short_traj = sorted(trajs, key=lambda t: len(t.history))[10]
# Use a short trajectory for a fast walkthrough.
sorted(trajs, key=lambda t: len(t.history))
print(f"Scoring trajectory: {short_traj.question_id}")
print(f"  Steps: {len(short_traj.history)}  |  mistake at step {short_traj.mistake_step}  ({short_traj.mistake_agent})")
print()
 
result = score_trajectory(
    trajectory    = short_traj,
    model         = model,
    tokenizer     = tokenizer,
    layer_variant = "lm_head",
    strategy      = "standard",
    device        = DEVICE,
    verbose       = False,
)

Scoring trajectory: 6b06d186921b8b390c65aebd0d16f09f60a47d2f1288ebe36953f734e84c0a3c
  Steps: 17  |  mistake at step 16  (WebSurfer)



In [29]:
print("Keys:", list(result.keys()))
print()
print(f"question_id  : {result['question_id']}")
print(f"true_step    : {result['true_step']}")
print(f"true_agent   : {result['true_agent']}")
print(f"skipped      : {result['skipped']}")
print()
print("Scores (step → S):")
for step_idx, score in sorted(result["scores"].items()):
    marker = "  ← MISTAKE" if step_idx == result["true_step"] else ""
    agent  = result["step_agents"][step_idx]
    print(f"  step {step_idx:2d}  [{agent:<30}]  {score:.6f}{marker}")
 
# %%
# Rank steps by score descending to see where the mistake step ranks.
ranked = sorted(result["scores"], key=result["scores"].__getitem__, reverse=True)
true_rank = ranked.index(result["true_step"]) + 1   # 1-indexed
print(f"\nMistake step {result['true_step']} ranks #{true_rank} out of {len(ranked)} steps.")

Keys: ['question_id', 'scores', 'true_step', 'true_agent', 'step_agents', 'skipped']

question_id  : 6b06d186921b8b390c65aebd0d16f09f60a47d2f1288ebe36953f734e84c0a3c
true_step    : 16
true_agent   : WebSurfer
skipped      : []

Scores (step → S):
  step  0  [human                         ]  0.000005
  step  1  [Orchestrator (thought)        ]  0.000003
  step  2  [Orchestrator (thought)        ]  0.000003
  step  3  [Orchestrator (-> WebSurfer)   ]  0.000001
  step  4  [WebSurfer                     ]  0.000002
  step  5  [Orchestrator (thought)        ]  0.000003
  step  6  [Orchestrator (-> WebSurfer)   ]  0.000001
  step  7  [Orchestrator (thought)        ]  0.000008
  step  8  [WebSurfer                     ]  0.000001
  step  9  [Orchestrator (thought)        ]  0.000003
  step 10  [Orchestrator (-> WebSurfer)   ]  0.000001
  step 11  [Orchestrator (thought)        ]  0.000008
  step 12  [WebSurfer                     ]  0.000001
  step 13  [Orchestrator (thought)        ]  0.0000